# Day 6 Demo 2b - Strands, hardened for production

Demo 2 gave you the ladder: call -> workflow -> agent -> agent+tools -> multi-agent -> multi-agent+tools.
Every rung there took the happy path. Real traffic does not.

This notebook re-walks three of those rungs and makes each robust:

- **Workflow** gains branching, an ambiguity handler, a human-escalation path, and exception handling.
- **Agent** gets tool guards, observability, bounded history, and the "tools first, model last" pattern.
- **Multi-agent** gets deterministic routing, an escalation tool, then a Swarm, with a decision table.

One idea runs through all of it: **every model call costs latency, money, and a chance to hallucinate. Spend them only where they earn their keep.** Deterministic code does the rest.

> Same anchor as Demo 1 and 2: TravelMind, PNR `JX48Q2`, flight `AI-302`. Real Bedrock only (Strands has no offline mode).

## Before you run

**Colab (3 steps)**
1. `!pip install strands-agents`
2. Provide AWS credentials: short-lived creds via env vars, or run `!aws configure`. Do not paste long-lived keys into the notebook.
3. Enable model access: Bedrock console -> Model catalog -> Claude Sonnet 4.5 in `us-west-2`.

**VS Code (3 steps)**
1. `pip install strands-agents` in your venv.
2. `aws configure` (or use a role); verify with `aws sts get-caller-identity`.
3. Confirm `REGION` and `MODEL` below point at a `us.` inference profile you have ENABLED.

In [ ]:
# %pip install strands-agents
from strands import Agent, tool
from strands.models import BedrockModel

REGION = "us-west-2"
MODEL  = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"   # a us. inference profile you have ENABLED

bedrock = BedrockModel(model_id=MODEL, region_name=REGION, temperature=0.2)
print("Strands model ready:", MODEL)

## 1 - Why the happy path is not enough

The Demo 2 workflow was `extractor -> writer`. It silently assumes:

- the customer gave a valid PNR,
- the intent is something TravelMind actually handles,
- every model call succeeds.

Break any one and the naive pipeline either answers with garbage or crashes. A production flow gives every outcome a home:

```
                       +-- valid PNR + standard intent ----> WRITER      (happy path)
   message --> EXTRACT +-- missing / unclear (recoverable) -> AMBIGUITY  (ask one question)
                       +-- non-standard intent ------------> ESCALATE    (red flag -> human)
                       +-- extractor / tool failure -------> ESCALATE    (fail safe)
```

Rules of the road for the rest of this notebook:

- **Validate, do not trust.** The model can claim a PNR; a regex decides if it is real.
- **Escalation is deterministic.** A red-flag handoff is a predictable, auditable ticket, not a model call.
- **Cheap first.** Try a rule before you reach for a model.

## 2 - Workflow v2: branch, recover, escalate, survive

### 2.1 The contract: structured extraction + business rules
`structured_output` returns a validated Pydantic object, not a string you have to parse. The field descriptions are the instructions the model reads.

In [ ]:
import re
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field, ValidationError

# Intents TravelMind supports. Anything else is "other" and gets escalated.
class Intent(str, Enum):
    BOOKING_STATUS = "booking_status"
    DISRUPTION     = "disruption"
    REBOOKING      = "rebooking"
    REFUND         = "refund"
    BAGGAGE        = "baggage"
    OTHER          = "other"

# What the extractor must return. Field descriptions are the model's instructions.
class Extraction(BaseModel):
    pnr: Optional[str] = Field(default=None, description="6-character alphanumeric PNR if the user gave one, else null")
    intent: Intent = Field(description="single best intent; use other for anything outside airline booking support")
    confidence: float = Field(ge=0.0, le=1.0, description="0 to 1 confidence in intent and pnr")
    ambiguity_reason: Optional[str] = Field(default=None, description="one short reason if unclear or multiple readings, else null")

# Business rule the model does NOT decide: a real PNR is 6 chars, mixed letters and digits.
PNR_RE = re.compile(r"^[A-Z0-9]{6}$")
def valid_pnr(pnr: Optional[str]) -> bool:
    if not pnr:
        return False
    p = pnr.strip().upper()
    return bool(PNR_RE.match(p)) and any(c.isdigit() for c in p) and any(c.isalpha() for c in p)

class Route(str, Enum):
    WRITE    = "write"
    CLARIFY  = "clarify"
    ESCALATE = "escalate"

def decide_route(ex: Extraction) -> Route:
    if ex.intent == Intent.OTHER:          # non-standard intent, red flag to a human
        return Route.ESCALATE
    if not valid_pnr(ex.pnr):              # standard ask, no usable PNR: try to recover it
        return Route.CLARIFY
    if ex.ambiguity_reason or ex.confidence < 0.6:
        return Route.CLARIFY
    return Route.WRITE

print("routing rules loaded")

### 2.2 Heuristics first: keep the model on the bench until you need it
A regex and a keyword table resolve the obvious traffic for **zero** model calls. The model is the fallback, not the default. The heuristic only fires when it is unambiguous (exactly one intent and a real PNR); when unsure it returns `None` and hands off. Cheap heuristics still need guardrails, or `FLIGHT` gets mistaken for a PNR.

In [ ]:
LLM_CALLS = {"n": 0}   # crude counter so you can SEE how often the model runs

INTENT_HINTS = {
    Intent.DISRUPTION:     ("cancel", "cancelled", "delayed", "delay", "disrupt", "stranded"),
    Intent.REBOOKING:      ("rebook", "another flight", "next flight", "change my flight"),
    Intent.REFUND:         ("refund", "money back", "reimburse"),
    Intent.BAGGAGE:        ("baggage", "luggage", "suitcase"),
    Intent.BOOKING_STATUS: ("status", "confirmed", "is my flight"),
}
CANDIDATE = re.compile(r"\b[A-Z0-9]{6}\b")

def find_pnr(text: str) -> Optional[str]:
    for tok in CANDIDATE.findall(text.upper()):
        if valid_pnr(tok):                 # requires mixed letters and digits, so FLIGHT is rejected
            return tok
    return None

def cheap_extract(msg: str) -> Optional[Extraction]:
    pnr = find_pnr(msg)
    hits = [i for i, words in INTENT_HINTS.items() if any(w in msg.lower() for w in words)]
    if len(hits) == 1 and valid_pnr(pnr):  # trust the rule only when it is unambiguous
        return Extraction(pnr=pnr, intent=hits[0], confidence=0.95, ambiguity_reason=None)
    return None                            # unsure -> let the model decide

def llm_extract(msg: str) -> Extraction:
    LLM_CALLS["n"] += 1
    extractor = Agent(model=bedrock, callback_handler=None,
        system_prompt=("You extract structured fields for TravelMind, an airline booking-exception assistant. "
                       "In-scope intents: booking_status, disruption, rebooking, refund, baggage. "
                       "Anything else is other. Never invent a PNR."))
    return extractor.structured_output(Extraction, msg)
    # newer SDKs also allow: extractor(msg, structured_output_model=Extraction).structured_output

def extract(msg: str) -> Extraction:
    return cheap_extract(msg) or llm_extract(msg)

LLM_CALLS["n"] = 0
print(extract("Is my flight confirmed? PNR JX48Q2"))
print("LLM calls used:", LLM_CALLS["n"])   # expect 0: the heuristic handled it

### 2.3 The guarded pipeline
Each stage runs inside `guard`, which catches any failure and records it. Any unrecoverable error routes to escalation with a safe message, so one bad model call never takes the whole request down. The output is a typed `Resolution` a backend can act on, not loose prose.

In [ ]:
SAFE_FALLBACK = ("Thanks for reaching out. I hit a snag pulling this up, so I am passing you "
                 "to a human agent who will follow up shortly.")

def escalate_to_human(customer_msg: str, ex: Optional[Extraction], reason: str):
    # deterministic and auditable. A red-flag handoff should be predictable, so no model call here.
    ticket = {
        "queue": "tier2_human",
        "reason": reason,
        "intent": ex.intent.value if ex else "unknown",
        "pnr": ex.pnr if ex else None,
        "customer_msg": customer_msg,
    }
    msg = ("This needs a specialist. I have raised it with our team and someone will get "
           "back to you shortly with next steps.")
    return msg, ticket

def guard(label, fn, trace):
    # run a stage, catch anything, record what happened. Returns (ok, value).
    try:
        v = fn()
        trace.append((label, "ok"))
        return True, v
    except ValidationError:
        trace.append((label, "validation_error"))
        return False, None
    except Exception as e:
        # in prod expect: EventLoopException, ContextWindowOverflowException, MaxTokensReachedException
        trace.append((label, "error:" + type(e).__name__))
        return False, None

class Resolution(BaseModel):
    route: Route
    customer_message: str
    internal_status: str
    escalation_ticket: Optional[dict] = None
    llm_calls: int = 0
    trace: list = Field(default_factory=list)

def _escalated(customer_msg, ex, reason, trace, status="escalated"):
    msg, ticket = escalate_to_human(customer_msg, ex, reason)
    return Resolution(route=Route.ESCALATE, customer_message=msg, internal_status=status,
                      escalation_ticket=ticket, llm_calls=LLM_CALLS["n"], trace=trace)

def handle(customer_msg: str) -> Resolution:
    LLM_CALLS["n"] = 0
    trace = []

    ok, ex = guard("extract", lambda: extract(customer_msg), trace)
    if not ok:
        return _escalated(customer_msg, None, "extractor_failed", trace, status="failed_safe")

    route = decide_route(ex)

    if route == Route.ESCALATE:
        return _escalated(customer_msg, ex, "non_standard_intent", trace)

    if route == Route.CLARIFY:
        def _clarify():
            LLM_CALLS["n"] += 1
            handler = Agent(model=bedrock, callback_handler=None,
                system_prompt=("You are TravelMind. The request is missing a valid PNR or is unclear. "
                               "Ask ONE friendly, specific question to get what you need. Do not guess."))
            return str(handler("Customer said: " + customer_msg + " || Extracted so far: " + ex.model_dump_json()))
        ok, msg = guard("clarify", _clarify, trace)
        if not ok:
            return _escalated(customer_msg, ex, "clarify_failed", trace, status="failed_safe")
        return Resolution(route=route, customer_message=msg, internal_status="awaiting_customer",
                          llm_calls=LLM_CALLS["n"], trace=trace)

    def _write():
        LLM_CALLS["n"] += 1
        writer = Agent(model=bedrock, callback_handler=None,
            system_prompt=("You are TravelMind. Write a warm, 2-sentence reply using the details given. "
                           "Do not invent flights or policies."))
        return str(writer("Details: " + ex.model_dump_json()))
    ok, msg = guard("write", _write, trace)
    if not ok:
        return _escalated(customer_msg, ex, "writer_failed", trace, status="failed_safe")
    return Resolution(route=route, customer_message=msg, internal_status="answered",
                      llm_calls=LLM_CALLS["n"], trace=trace)

print("pipeline ready")

### 2.4 Run every path
Four crafted inputs hit the four outcomes, plus a forced failure to prove the fail-safe. Watch `llm_calls`: the happy path spends one (only the writer, extraction was free); escalation spends one for classification and zero for the handoff.

In [ ]:
tests = [
    ("happy",        "Is my flight confirmed? PNR JX48Q2"),
    ("missing pnr",  "My flight got cancelled, what are my options?"),
    ("ambiguous",    "I have two bookings and one is a mess, can you sort it out?"),
    ("non-standard", "I want to file a legal complaint and speak to your CEO about this airline"),
]
for label, msg in tests:
    r = handle(msg)
    print("[" + label + "] route=" + r.route.value + "  llm_calls=" + str(r.llm_calls) + "  status=" + r.internal_status)
    print("   reply:", r.customer_message)
    if r.escalation_ticket:
        print("   ticket:", r.escalation_ticket)
    print("   trace:", r.trace)
    print()

# fail-safe: force the extractor to blow up, prove the pipeline never crashes
_real_extract = extract
def _boom(_):
    raise RuntimeError("bedrock unavailable")
extract = _boom
print("[forced failure] status =", handle("Is my flight confirmed? PNR JX48Q2").internal_status)
extract = _real_extract

**Best practices**
- One reason per route. `decide_route` is pure and testable without a model.
- Keep the writer honest: it phrases the extracted facts here. Real lookups arrive in section 4.
- The trace is your black box: log route, status, and llm_calls on every request.

**What changes in production**
- Replace the in-line escalation with a real ticket to a queue (SNS/SQS, Zendesk, PagerDuty), keyed by request id.
- Catch the named Strands errors explicitly: `EventLoopException`, `ContextWindowOverflowException`, `MaxTokensReachedException`, plus pydantic `ValidationError`. Strands already retries throttling for you with backoff.
- No hardcoded region, model id, or secrets: read them from config or SSM. Use an IAM role with least privilege on `bedrock:InvokeModel`.
- Emit metrics (latency, route mix, escalation rate, tokens) to CloudWatch; alert on escalation-rate spikes.

## 3 - Workflow v3: the same routing as a declarative Graph

Plain Python branching is perfect and you should not feel bad about it. When the business wants the routing to be *inspectable* (which node ran, in what order, why), a `Graph` encodes the same rules as nodes and conditional edges. The condition functions read `GraphState.results` and reuse your `decide_route`.

In [ ]:
from strands.multiagent import GraphBuilder

# The extractor node returns STRICT JSON so an edge condition can read and validate it.
g_extractor = Agent(model=bedrock, callback_handler=None,
    system_prompt=(
        "Extract fields for TravelMind and return ONLY minified JSON, no prose. "
        "Keys: pnr (string or null), intent (one of booking_status, disruption, rebooking, "
        "refund, baggage, other), confidence (number 0 to 1), ambiguity_reason (string or null). "
        "Use other for anything outside airline booking support. Never invent a PNR."))

g_writer = Agent(model=bedrock,
    system_prompt="You are TravelMind. Write a warm 2-sentence reply from the JSON details. Do not invent flights.")
g_clarifier = Agent(model=bedrock,
    system_prompt="You are TravelMind. The PNR is missing or invalid, or the ask is unclear. Ask ONE specific question.")
g_escalation = Agent(model=bedrock,
    system_prompt="You are TravelMind. Say the issue is being handed to a human specialist, in 2 warm sentences.")
# In production this escalation node is a tool that writes a ticket to a queue, not a model call.

def _extraction_from_state(state):
    node = state.results.get("extract")
    if not node:
        return None
    try:
        return Extraction.model_validate_json(str(node.result))   # pydantic gives typing back inside the graph
    except Exception:
        return None

def route_is(target: Route):
    def _cond(state):
        ex = _extraction_from_state(state)
        if ex is None:
            return target == Route.ESCALATE      # unparseable extraction -> escalate
        return decide_route(ex) == target
    return _cond

builder = GraphBuilder()
builder.add_node(g_extractor,  "extract")
builder.add_node(g_writer,     "write")
builder.add_node(g_clarifier,  "clarify")
builder.add_node(g_escalation, "escalate")
builder.add_edge("extract", "write",    condition=route_is(Route.WRITE))
builder.add_edge("extract", "clarify",  condition=route_is(Route.CLARIFY))
builder.add_edge("extract", "escalate", condition=route_is(Route.ESCALATE))
builder.set_entry_point("extract")
graph = builder.build()

def run_graph(msg):
    print(">>>", msg)
    try:
        graph(msg)                          # nodes stream their reply; exactly one branch fires
    except Exception as e:
        print("   graph failed safe ->", type(e).__name__)

run_graph("Is my flight confirmed? PNR JX48Q2")                     # -> write
print("----")
run_graph("I want to file a legal complaint and speak to your CEO") # -> escalate

**Plain Python vs Graph**

| | Plain Python (2.3) | Graph (3) |
|---|---|---|
| Setup | none | GraphBuilder, nodes, edges |
| Control | total, line by line | declarative rules |
| Debugging | trivial, it is just code | trace shows node order |
| Parallel steps | you wire threads yourself | free, independent nodes run together |
| Reuse | copy the function | drop the graph in as a node elsewhere |
| Best when | simple, few branches | many branches, approvals, quality gates, audits |

Skeptic's note: a Graph does not make wrong routing right. Your `decide_route` still owns the logic; the Graph just makes the flow visible.

## 4 - Agent v2: tools first, tight control, minimal spend

An autonomous tool-using agent is convenient and expensive: each turn is a model call. Two levers matter in production: make tools fail safely, and stop asking the model to decide things you already know.

### 4.1 Hardened tools + a bounded autonomous agent
- `lookup_booking` returns `found=false` instead of raising, and rejects a bad PNR before touching the backend.
- `flaky_fees` raises on purpose. Strands feeds the error back to the model as a tool result, so the loop survives and the model copes.
- `TOOL_LOG` records every call. `SlidingWindowConversationManager` caps history so a long chat cannot blow the context window.

In [ ]:
from strands import Agent, tool
from strands.agent.conversation_manager import SlidingWindowConversationManager

TOOL_LOG = []
_DB = {"JX48Q2": {"status": "CANCELLED", "flight": "AI-302", "date": "2026-06-12"}}

@tool
def lookup_booking(pnr: str) -> dict:
    """Look up a booking by its 6-character PNR. Returns found=false for unknown or malformed PNRs."""
    TOOL_LOG.append(("lookup_booking", pnr))
    p = (pnr or "").strip().upper()
    if not valid_pnr(p):
        return {"found": False, "error": "invalid_pnr_format", "pnr": p}   # control: stop before the backend
    rec = _DB.get(p)
    if not rec:
        return {"found": False, "error": "not_found", "pnr": p}            # graceful, not an exception
    return {"found": True, "pnr": p, **rec}

@tool
def get_rebooking_options(pnr: str) -> list:
    """Alternative flights for a disrupted booking. Call only after a successful lookup_booking."""
    TOOL_LOG.append(("get_rebooking_options", pnr))
    return [{"flight": "AI-318", "dep": "18:40"}, {"flight": "6E-552", "dep": "21:15"}]

@tool
def flaky_fees(pnr: str) -> dict:
    """Look up change fees. Demo tool that raises, to show the agent loop survives a tool error."""
    TOOL_LOG.append(("flaky_fees", pnr))
    raise RuntimeError("fees service timeout")   # Strands returns this to the model as an error result

travel_agent = Agent(
    model=bedrock,
    tools=[lookup_booking, get_rebooking_options, flaky_fees],
    system_prompt=("You are TravelMind. Always call lookup_booking BEFORE answering. "
                   "If a tool returns found=false, ask the customer to recheck their PNR. "
                   "Never invent a PNR, flight, or fee."),
    conversation_manager=SlidingWindowConversationManager(window_size=20),   # bound history growth
)

TOOL_LOG.clear()
print(travel_agent("My flight on JX48Q2 was cancelled. What are my rebooking options and any change fees?"))
print("tools called:", TOOL_LOG)

### 4.2 "Tools first, model last"
If you already know the first move is "look up the booking", do it in code. Reserve the model for the one thing it is good at here: phrasing. Compare `model_calls`: the autonomous agent above used several; this uses one.

In [ ]:
import json

# The autonomous agent above spent several model calls (decide, tool, decide, answer).
# When you already know the first step, do it in code and spend the model only on phrasing.
def resolve_disruption(pnr: str) -> dict:
    TOOL_LOG.clear()
    booking = lookup_booking(pnr)                 # deterministic, 0 model calls
    if not booking["found"]:
        return {"model_calls": 0, "tools": list(TOOL_LOG),
                "reply": "I could not find that PNR. Could you double-check the 6-character code?"}
    options = get_rebooking_options(pnr)          # deterministic, 0 model calls
    phraser = Agent(model=bedrock, callback_handler=None,
        system_prompt="Write ONE warm, 2-sentence reply. Use only the facts given. Do not add flights or fees.")
    reply = str(phraser(json.dumps({"booking": booking, "options": options})))   # the ONLY model call
    return {"model_calls": 1, "tools": list(TOOL_LOG), "reply": reply}

print(resolve_disruption("JX48Q2"))

**Controlling tool calls**
- A guard inside the tool (validate, deny, cache) is the simplest gate and always runs.
- SDK-native gate: a `BeforeToolCallEvent` hook can approve or cancel a call before it runs.

```python
from strands.hooks import BeforeToolCallEvent
def gate(event: BeforeToolCallEvent):
    if event.tool_use["name"] == "flaky_fees":
        event.cancel_tool = "fees disabled"     # deny this tool
agent = Agent(model=bedrock, tools=[lookup_booking, flaky_fees], hooks=[gate])
```

- Forcing a tool: Bedrock `toolChoice` can force tool use, but not every model supports every mode. Prefer a strong system prompt plus a wrapper that checks the tool actually ran.

**Spending the model wisely**
- Deterministic routing and extraction beat a model call for anything rule-shaped.
- Set `max_tokens` on `BedrockModel`; keep tool outputs small (return summaries, not dumps).
- Let the model orchestrate only when the next step genuinely depends on judgement it alone has.

## 5 - Multi-agent v2: route deterministically, escalate explicitly

Demo 2 gave the orchestrator every query. Cheaper: a keyword router sends clear cases straight to a specialist and only wakes the orchestrator model for genuinely mixed requests. Out-of-scope asks hit an `escalate` tool.

In [ ]:
ROUTING_LOG = []

billing_agent = Agent(model=bedrock, callback_handler=None,
    system_prompt="Billing specialist. Be precise about refunds and fees. 2 to 3 sentences.")
disruption_agent = Agent(model=bedrock, callback_handler=None,
    tools=[lookup_booking, get_rebooking_options],
    system_prompt="Flight-disruption specialist. Look up the booking before answering. Never invent details.")

@tool
def handle_billing(question: str) -> str:
    """Refunds, fees, and charges."""
    return str(billing_agent(question))

@tool
def handle_disruption(question: str) -> str:
    """Delays, cancellations, rebooking. Uses the booking tools."""
    return str(disruption_agent(question))

@tool
def escalate(question: str) -> str:
    """Out-of-scope or non-standard requests, routed to a human queue."""
    ROUTING_LOG.append("escalate")
    return "Raised to a human specialist; the customer will be contacted shortly."

orchestrator = Agent(model=bedrock,
    tools=[handle_billing, handle_disruption, escalate],
    system_prompt=("Route each query to the right specialist tool, then reply warmly in under 60 words. "
                   "Use escalate for anything outside billing or flight disruption."))

def route_keyword(msg: str):
    m = msg.lower()
    bill = any(w in m for w in ("refund", "fee", "charge", "money"))
    disr = any(w in m for w in ("cancel", "delay", "rebook", "disrupt", "stranded"))
    if bill and not disr:
        return "billing"
    if disr and not bill:
        return "disruption"
    return None    # mixed or unclear -> the orchestrator decides

def support(msg: str) -> str:
    ROUTING_LOG.clear()
    direct = route_keyword(msg)
    if direct == "billing":
        ROUTING_LOG.append("billing(direct)")
        return str(billing_agent(msg))
    if direct == "disruption":
        ROUTING_LOG.append("disruption(direct)")
        return str(disruption_agent(msg))
    ROUTING_LOG.append("orchestrator(llm)")
    return str(orchestrator(msg))

print(support("My flight JX48Q2 was cancelled, what are my options?"))            # disruption only -> direct
print("route:", ROUTING_LOG)
print("----")
print(support("I want a refund for the change fees on my cancelled flight JX48Q2"))  # mixed -> orchestrator
print("route:", ROUTING_LOG)

**Best practices**
- Specialists stay single-domain; the router owns coordination.
- `ROUTING_LOG` shows which path served each request, direct or via the orchestrator.
- The disruption specialist carries the booking tools; billing does not. Give each agent only the tools its job needs.

**What changes in production**
- The `escalate` tool writes a real ticket and returns its id.
- Bound nesting depth and log each hop; a tool-using specialist wrapped as a tool is nested loops.

## 6 - Swarm, and how to choose

A Swarm lets agents hand off to each other autonomously through shared context. You give up ordering control and get self-organisation. Each agent needs a unique `name`.

In [ ]:
from strands.multiagent import Swarm

triage = Agent(model=bedrock, name="triage",
    system_prompt="Triage airline support. Hand off to the right specialist. Do not answer directly.")
disruption_specialist = Agent(model=bedrock, name="disruption",
    tools=[lookup_booking, get_rebooking_options],
    system_prompt="Flight-disruption specialist. Look up the booking, then resolve. Hand back when done.")
billing_specialist = Agent(model=bedrock, name="billing",
    system_prompt="Billing specialist for refunds and fees.")

swarm = Swarm([triage, disruption_specialist, billing_specialist])   # each name must be UNIQUE
print(swarm("My flight JX48Q2 was cancelled, what are my options?"))

**Four ways to orchestrate the same support flow**

| Pattern | Control | Deterministic | Observability | Model spend | Reach for it when | Avoid when |
|---|---|---|---|---|---|---|
| Heuristic + plain Python (2) | total | yes | your logs | lowest | rules cover most traffic | logic is genuinely fuzzy |
| Agents as tools (5) | high | mostly | per-tool logs | medium | one lead delegates to specialists | steps must run in a fixed audited order |
| Graph (3) | high | yes | built-in trace | medium | approvals, quality gates, branches, parallel steps | a two-line if/else would do |
| Swarm (6) | low | no | debug logs | highest | the sequence cannot be known upfront | you need repeatable, auditable order |

Rule of thumb: **start with the cheapest pattern that fits, and let the shape of the problem pull you up the list, not the other way around.**

## Recap

- Branch on structured, validated output. Give every unhappy path a home.
- Escalation and routing that can be rules should be rules. The model is your most expensive component.
- Tools first, model last. Guard tools, bound loops, log everything.
- Pick the lightest orchestration that fits; climb only when the problem makes you.

Next, **Demo 3** hosts one of these hardened agents on **AgentCore**.